In [ ]:
!pip install datasets sentence-transformers pyarrow

In [ ]:
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from google.colab import files # For downloading

In [ ]:
# --- 1. Load the Model to the GPU ---
print("Loading the SentenceTransformer model to CUDA...")
model = SentenceTransformer(
  "thomas-sounack/BioClinical-ModernBERT-base",
  device="cuda"  # Use the GPU
)
print("Model loaded.")

# --- 2. Load a Subset of the Dataset object ---
# We select the first 140,000 examples from the training split.
subset_size = 140000
hf_dataset = load_dataset("louisbrulenaudet/clinical-trials", split=f"train[:{subset_size}]")

# --- 3. Define the Embedding Function and Apply it with .map() ---
columns_to_embed = [
  "brief_summary",
  "eligibility_criteria"
]

def embed_texts(batch):
  for col in columns_to_embed:
    # No tolist() needed, model.encode returns numpy arrays which map() handles
    embeddings = model.encode(batch[col])
    batch[f"{col}_embedding"] = embeddings
  return batch

print(f"\nGenerating embeddings for a subset of {subset_size} examples using .map() on GPU...")
# Use a larger batch size for GPU
embedded_dataset = hf_dataset.map(
  embed_texts,
  batched=True,
  batch_size=256,
  desc="Embedding columns"
)
print("Embedding generation complete.")

# --- 4. Convert to Pandas and Save to Parquet ---
print("\nConverting to Pandas DataFrame...")
full_df_with_embeddings = embedded_dataset.to_pandas()
print("Conversion complete.")

print("\nSaving DataFrame to Parquet format...")
# This saves the file in the Colab virtual machine's local storage.
full_df_with_embeddings.to_parquet('subset_dataset_with_embeddings.parquet')
print("File saved.")

# --- 5. Download the File ---
print("\nTriggering download. This will take some time...")
# This command tells your browser to download the file from the Colab machine.
files.download('subset_dataset_with_embeddings.parquet')

Loading the SentenceTransformer model to CUDA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Embedding columns:   0%|          | 0/140000 [00:00<?, ? examples/s]

W1101 19:22:04.414000 415 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Embedding generation complete.

Converting to Pandas DataFrame...
Conversion complete.

Saving DataFrame to Parquet format...
File saved.

Triggering download. This will take some time...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>